In [1]:
%cd ../../
%load_ext dotenv
%dotenv

/Users/hoangle/Projects/untangling-people/ylva/fwo_models


In [2]:
import time
import random
import json
import uuid
from pathlib import Path
from itertools import combinations, product

import pandas as pd
import polars as pl

In [3]:
seed = time.time()
random.seed(seed)

In [4]:
restaurant = 3
schoolyear = "25-26"
date_start = "2025-11-03"

# Load tables

In [5]:
path = "data/processed/dim_meal_types.xlsx"

dim_meal_types = pl.read_excel(path)
dim_meal_types

meal_type_id,meal_type,meal_type_en
i64,str,str
1,"""Kala""","""fish"""
2,"""Liha""","""meat"""
3,"""Kana""","""chicken"""
4,"""Vegaani""","""vegan"""
5,"""Kasvis""","""vegetarian"""
6,"""Buffet""","""buffet"""
7,"""Not Mapped""","""not_mapped"""


In [6]:
path = "data/processed/dim_meals_1029.parquet"
dim_meals = (
    pl.read_parquet(path)
    # .drop('meal_codes', 'names', 'src')

    # .join(
    #     dim_meal_types.select('meal_type_id', 'meal_type_en'),
    #     left_on='meal_type', right_on='meal_type_id',
    #     how='left'
    # )
)
dim_meals.head()

id,meal_codes,names,restaurants,meal_type,schoolyear,attributes,co2,src
i64,list[i64],list[str],list[i64],i64,str,list[str],f32,list[str]
0,[90000000],"[""Kikhernetaginea& syysomenajogurttia""]",[1],5,"""23-24""",[],0.41,"[""pos_Jan23-Oct24""]"
1,[90000001],"[""Rapea Meiramikana""]","[1, 4]",3,"""23-24""",[],1.26,"[""pos_Jan23-Oct24""]"
2,[7203],"[""Kasvismuhennos Caponata""]","[1, 2, 4]",4,"""23-24""",[],0.42,"[""pos_Jan23-Oct24"", ""menus_meals""]"
3,[9058],"[""TexMex-siemenpyöryköitä ja Arrabiattakastiketta"", ""TexMex-siemenpyöryköitä ja Arrabiattakastiket""]","[1, 2, 4]",4,"""24-25""","[""gluten_free"", ""vegan-kpl""]",0.56,"[""pos_Jan23-Oct24"", ""menus_meals"", … ""menus_week1-6""]"
4,[6877],"[""Kasvisjalapenonuggetteja, tomaattisalsaa"", ""Kasvis-jalapnuget ja tomatsals""]","[1, 2, 4]",4,"""24-25""",[],0.44,"[""pos_Jan23-Oct24"", ""menus_meals"", ""pos_Nov24-Mar25""]"


In [7]:
path = "data/processed/pos/forecasted/1030.xlsx"

schema = {
    'meal_id': pl.Int64,
    'restaurant_id': pl.Int64,
    'pcs': pl.Int64,
}

pos_forecasted = pl.read_excel(path, schema_overrides=schema)
pos_forecasted.head()

meal_id,restaurant_id,pcs
i64,i64,i64
229,4,65
98,2,74
47,1,36
81,4,21
316,3,7


In [8]:
path = Path("data/processed/pos_daily/forecasted") / f"{restaurant}.xlsx"

schema = {
    'date': pl.Date,
    'restaurant_id': pl.Int64,
    'pcs': pl.Int64,
}

pos_restaurant_forecasted = pl.read_excel(path, schema_overrides=schema)
pos_restaurant_forecasted.head()

date,restaurant_id,pcs
date,i64,i64
2025-11-03,3,231
2025-11-04,3,263
2025-11-05,3,249
2025-11-06,3,260
2025-11-07,3,192


In [9]:
path = Path("data/processed/waste_daily/forecasted") / f"{restaurant}.xlsx"

schema = {
    'date': pl.Date,
    'restaurant_id': pl.Int64,
    'waste': pl.Float64,
}

waste_restaurant_forecasted = pl.read_excel(path, schema_overrides=schema)
waste_restaurant_forecasted.head()

date,restaurant_id,waste
date,i64,f64
2025-11-03,3,14.068347
2025-11-04,3,15.47766
2025-11-05,3,15.233518
2025-11-06,3,17.024495
2025-11-07,3,15.358867


### Load list of selected meals specific for each restaurant and weekday

File `recommended_meal_list.json` has following structure

```json
{
    "<restaurant_id>": {
        "<weekday_id>": {
            "non-vegan": [<list of meal id>],
            "vegan": [<list of meal id>],
        }
    }
}
```

In [10]:
path = "data/processed/recommended_meal_list.json"
with open(path) as file:
    meals_by_day_raw = json.load(file)

meals_by_day = {}
match restaurant:
    case 1:
        for restau, weekdays in meals_by_day_raw.items():
            meals_by_day[int(restau)] = {}
            for weekday, meals in weekdays.items():
                meals_by_day[int(restau)][int(weekday)] = meals
    case _:
        dishes_non_vegan = (
            dim_meals
            .filter(
                (1 == 1)
                & (pl.col('restaurants').list.contains(restaurant))
                & (pl.col('meal_type').is_in([1, 2, 3]))
            )
            ['id']
            .to_list()
        )  # fmt: skip
        dishes_vegan = (
            dim_meals
            .filter(
                (1 == 1)
                & (pl.col('restaurants').list.contains(restaurant))
                & (pl.col('meal_type').is_in([4, 5]))
            )
            ['id']
            .to_list()
        )  # fmt: skip

        meals_by_day[restaurant] = {}
        for week in range(1, 6):
            meals_by_day[restaurant][week] = {
                "non-vegan": dishes_non_vegan,
                "vegan": dishes_vegan,
            }

# Craft menus

In [11]:
NUM_VEGAN_PER_DAY = 2
NUM_MEALS_PER_DAY = 3
MAX_MEAL_OCCURENCES = 2
NUM_FISH_PER_WEEK = 2

NUM_DAY_LEVEL_MENUS = 1_000_000
NUM_WEEK_LEVEL_MENUS = 1000
NUM_MENUS_FINAL = 20

THETA_CO2 = 0.5
THETA_WASTE = 0.04
THETA_KELA = 2
THETA_GLUTEN = 1

ALPHA_POS = 2
ALPHA_CO2 = 1
ALPHA_WASTE = 1
ALPHA_KELA = 2
ALPHA_GLUTEN = 2

## Craft the day-level menus

Day-level menus must satisfy:
- condition (2), (6) and (9)
- containing a variety of meals

***[June 3, 2025]***
- Condition (1.2) and (8) changed from strict to loose

In [12]:
def craft_day_level_menu(meals_by_day: dict, restaurant: int, weekday: int, n_max: int = 10_000_000) -> list:
    assert restaurant in meals_by_day and weekday in meals_by_day[restaurant]
    non_vegan = meals_by_day[restaurant][weekday]['non-vegan']
    vegan = meals_by_day[restaurant][weekday]['vegan']

    combos = [[x[0], *x[1]] for x in product(non_vegan, combinations(vegan, NUM_VEGAN_PER_DAY))]
    
    return combos[:n_max]

In [13]:
combos_mon = craft_day_level_menu(meals_by_day, restaurant, 1)
combos_tue = craft_day_level_menu(meals_by_day, restaurant, 2)
combos_wed = craft_day_level_menu(meals_by_day, restaurant, 3)
combos_thu = craft_day_level_menu(meals_by_day, restaurant, 4)
combos_fri = craft_day_level_menu(meals_by_day, restaurant, 5)

# Craft week-level menu

Week-level menus must satisfy:
- condition (1.1), (3)

In [14]:
meal_type_fish = dim_meal_types.filter(pl.col('meal_type_en') == pl.lit('fish'))['meal_type_id'].head().item()

# Create table containing week menu candidates
menus_week = pl.DataFrame({
    '1': random.choices(combos_mon, k=NUM_DAY_LEVEL_MENUS),
    '2': random.choices(combos_tue, k=NUM_DAY_LEVEL_MENUS),
    '3': random.choices(combos_wed, k=NUM_DAY_LEVEL_MENUS),
    '4': random.choices(combos_thu, k=NUM_DAY_LEVEL_MENUS),
    '5': random.choices(combos_fri, k=NUM_DAY_LEVEL_MENUS),
    'weeklevel_idx': pl.Series([str(uuid.uuid4()) for _ in range(NUM_DAY_LEVEL_MENUS)])
})

menus_week.head()

1,2,3,4,5,weeklevel_idx
list[i64],list[i64],list[i64],list[i64],list[i64],str
"[343, 426, 893]","[527, 447, 516]","[398, 368, 520]","[413, 425, 509]","[361, 395, 519]","""ad40fcda-47fe-4f6e-8867-71d1e8…"
"[369, 96, 395]","[428, 280, 523]","[557, 96, 321]","[441, 426, 508]","[438, 96, 521]","""ba0fdd9e-2d50-4743-9698-7fbe28…"
"[440, 498, 519]","[441, 254, 525]","[25, 500, 521]","[335, 102, 496]","[526, 321, 515]","""fd5cf126-73e0-4082-abf9-44c2ac…"
"[398, 226, 513]","[139, 513, 525]","[331, 11, 254]","[369, 524, 893]","[438, 226, 508]","""94065cfb-3fdc-453b-9341-b69ab4…"
"[143, 500, 893]","[413, 409, 495]","[441, 513, 522]","[438, 509, 519]","[438, 500, 521]","""ad40f5ab-f148-41c0-a14b-7d4319…"


In [15]:
ids_valid_week_menu = (
    menus_week
    .select(
        'weeklevel_idx',
        pl.concat_list(["1", "2", "3", "4", "5"]).alias('meal')
    )
    .explode('meal')


    # For each week menu, find the max occurence of meals in the week menu
    .with_columns(
        pl.len().over('weeklevel_idx', 'meal').alias('count_occurence'),
    )
    .with_columns(
        pl.col('count_occurence').max().over('weeklevel_idx').alias('count_max_occurence')
    )

    # Remove week menu candidates not satisfying (3)
    .filter(pl.col('count_max_occurence') <= MAX_MEAL_OCCURENCES)
    


    # Add meal type info and count no. fish meals of each week menu candidate
    .join(dim_meals.select('id', 'meal_type'), left_on='meal', right_on='id', how='left')
    .with_columns(
        (pl.col('meal_type') == meal_type_fish).cast(pl.Int32).alias('is_fish')
    )
    .with_columns(
        pl.col('is_fish').sum().over('weeklevel_idx').alias('count_fish')
    )

    # Remove week menu candidates not satisfying (1.1)
    .filter(pl.col('count_fish') >= NUM_FISH_PER_WEEK)

    .select('weeklevel_idx')
    .unique()
)

menus_week = (
    menus_week
    .join(ids_valid_week_menu, on='weeklevel_idx', how='inner')
    .sample(NUM_WEEK_LEVEL_MENUS)

    .melt(
        'weeklevel_idx',
        value_vars=["1", "2", "3", "4", "5"],
        variable_name="weekday",
        value_name="meal"
    )

    .with_columns(pl.col('weekday').cast(pl.Int32))
)
menus_week.head()

/var/folders/pr/8dv_cj95295bxt_hr8hzrmk40000gn/T/ipykernel_8636/4179382572.py:44: DeprecationWarning: `DataFrame.melt` is deprecated; use `DataFrame.unpivot` instead, with `index` instead of `id_vars` and `on` instead of `value_vars`
  .melt(


weeklevel_idx,weekday,meal
str,i32,list[i64]
"""cf4995e1-48a1-467e-8db8-01b32a…",1,"[904, 102, 348]"
"""ddf6a7fe-0851-4936-893d-fe3547…",1,"[371, 515, 519]"
"""b41ae23b-9999-414b-977b-54c5af…",1,"[331, 395, 519]"
"""2de60022-57db-4f83-986c-6d1d7a…",1,"[316, 321, 409]"
"""10de70b4-fc7c-4f5f-a36d-9231f2…",1,"[413, 96, 348]"


# Add meal-specific info and date-specific needed for calculating score

Following info will be added:
- `whole_pos` (forecasted)
- `whole_waste` (forecasted)
- meal's CO2
- meal's POS (forecasted)
- meal's gluten
- meal's kela

In [16]:
weekday2date = pl.DataFrame({
    'weekday': [1, 2, 3, 4, 5],
    'date': pl.Series(pd.date_range(date_start, periods=5)).dt.date()
})
weekday2date.head()

weekday,date
i64,date
1,2025-11-03
2,2025-11-04
3,2025-11-05
4,2025-11-06
5,2025-11-07


In [17]:
s_gluten = "gluten_free"
s_kela = "kela"

menus_week = (
    menus_week
    .with_columns(pl.lit(restaurant).alias('restaurant_id'))
    .explode('meal')

    .join(weekday2date, on='weekday')


    .join(
        pos_forecasted,
        left_on=['meal', 'restaurant_id'],
        right_on=['meal_id', 'restaurant_id'],
        how='left'
    )
    .rename({'pcs': 'pos'})
    .join(pos_restaurant_forecasted, on=['date', 'restaurant_id'], how='left')
    .rename({'pcs': 'whole_pos'})
    .join(waste_restaurant_forecasted, on=['date', 'restaurant_id'], how='left')
    .rename({'waste': 'whole_waste'})

    .join(
        dim_meals.select(
            'id', 'co2',
            pl.col('attributes').list.contains(s_gluten).alias('is_gluten').cast(pl.Int32),
            pl.col('attributes').list.contains(s_kela).alias('is_kela').cast(pl.Int32),
        ),
        left_on='meal', right_on='id', how='left'
    )
)

menus_week.head()

weeklevel_idx,weekday,meal,restaurant_id,date,pos,whole_pos,whole_waste,co2,is_gluten,is_kela
str,i32,i64,i32,date,i64,i64,f64,f32,i32,i32
"""cf4995e1-48a1-467e-8db8-01b32a…",1,904,3,2025-11-03,12,231,14.068347,0.0,0,0
"""cf4995e1-48a1-467e-8db8-01b32a…",1,102,3,2025-11-03,1,231,14.068347,0.82,0,0
"""cf4995e1-48a1-467e-8db8-01b32a…",1,348,3,2025-11-03,5,231,14.068347,0.43,0,0
"""ddf6a7fe-0851-4936-893d-fe3547…",1,371,3,2025-11-03,2,231,14.068347,0.8,1,1
"""ddf6a7fe-0851-4936-893d-fe3547…",1,515,3,2025-11-03,null,231,14.068347,0.44,0,0


In [20]:
if restaurant == 3:
    menus_week = menus_week.with_columns(pl.col('pos').fill_null(10))
else:
    menus_invalid = menus_week.filter(pl.col('pos').is_null()).select('weeklevel_idx').unique()
    menus_week = menus_week.join(menus_invalid, on='weeklevel_idx', how='anti')

In [19]:
menus_week.filter(pl.col('pos').is_null())

weeklevel_idx,weekday,meal,restaurant_id,date,pos,whole_pos,whole_waste,co2,is_gluten,is_kela
str,i32,i64,i32,date,i64,i64,f64,f32,i32,i32
"""ddf6a7fe-0851-4936-893d-fe3547…",1,515,3,2025-11-03,null,231,14.068347,0.44,0,0
"""ddf6a7fe-0851-4936-893d-fe3547…",1,519,3,2025-11-03,null,231,14.068347,0.83,0,0
"""b41ae23b-9999-414b-977b-54c5af…",1,519,3,2025-11-03,null,231,14.068347,0.83,0,0
"""2de60022-57db-4f83-986c-6d1d7a…",1,409,3,2025-11-03,null,231,14.068347,1.05,1,1
"""10de70b4-fc7c-4f5f-a36d-9231f2…",1,413,3,2025-11-03,null,231,14.068347,0.67,0,1
…,…,…,…,…,…,…,…,…,…,…
"""4507314d-d610-4e49-b4c7-c4bd95…",5,514,3,2025-11-07,null,192,15.358867,0.61,0,0
"""4507314d-d610-4e49-b4c7-c4bd95…",5,447,3,2025-11-07,null,192,15.358867,0.7,0,1
"""4507314d-d610-4e49-b4c7-c4bd95…",5,511,3,2025-11-07,null,192,15.358867,0.43,0,0


In [19]:
menus_week

weeklevel_idx,weekday,meal,restaurant_id,date,pos,whole_pos,whole_waste,co2,is_gluten,is_kela
str,i32,i64,i32,date,i64,i64,f64,f32,i32,i32


# Calculate fitness value

In [24]:
menus_week = (
    menus_week

    # Calculate score for each date
    .group_by('weeklevel_idx', 'weekday')
    .agg(
        pl.concat_list(pl.struct('meal', 'pos')).flatten().alias('meals_planned'),

        pl.col('pos').sum().alias('sum_pos'),
        (pl.col('pos') * pl.col('co2')).sum().alias('sum_co2_pos'),
        pl.col('is_gluten').sum().alias('sum_gluten'),
        pl.col('is_kela').sum().alias('sum_kela'),

        pl.col('whole_pos').first(),
        pl.col('whole_waste').first(),
    )

    .with_columns(
        (
            ALPHA_POS * (pl.col('sum_pos') / pl.col('whole_pos') - 1).abs()
            + ALPHA_CO2 * (pl.col('sum_co2_pos') / pl.col('sum_pos') / THETA_CO2)
            + ALPHA_WASTE * (pl.col('whole_waste') / pl.col('sum_pos') / THETA_WASTE)
            + ALPHA_GLUTEN * (1 - pl.col('sum_gluten') / THETA_GLUTEN).clip(0)
            + ALPHA_KELA * (1 - pl.col('sum_kela') / THETA_GLUTEN).clip(0)
        ).alias('score_day')
    )

    # Calculate score for entire week
    .with_columns(
        pl.col('score_day').sum().over('weeklevel_idx').alias('score_week')
    )
    .with_columns(
        pl.col('score_week').rank('dense', descending=False).over(partition_by=['weeklevel_idx']).alias('rank')
    )
    .filter(pl.col('rank') <= NUM_MENUS_FINAL)
    .sort('rank')


    # Keep columns as data model
    .join(weekday2date, on='weekday', how='left')
    .select(
        pl.concat_str(
            [
                pl.col('weeklevel_idx'),
                pl.col('date').dt.strftime(r"%Y-%m-%d"), 
                pl.lit(restaurant)
            ],
            separator='|'
        ).alias('id'),
        pl.lit(restaurant).alias('restaurant'),
        'weeklevel_idx',
        'date',
        'whole_waste',
        'whole_pos',
        'score_week',
        'meals_planned'
    )

    .sort('score_week')
)

menus_week.head()

id,restaurant,weeklevel_idx,date,whole_waste,whole_pos,score_week,meals_planned
str,i32,str,date,f64,i64,f64,list[struct[2]]
"""760c9d80-aa9f-4963-9514-34227c…",1,"""760c9d80-aa9f-4963-9514-34227c…",2025-11-05,37.911414,1064,19.640321,"[{20,226}, {39,438}, {252,120}]"
"""760c9d80-aa9f-4963-9514-34227c…",1,"""760c9d80-aa9f-4963-9514-34227c…",2025-11-04,34.40957,1054,19.640321,"[{528,91}, {68,450}, {114,294}]"
"""760c9d80-aa9f-4963-9514-34227c…",1,"""760c9d80-aa9f-4963-9514-34227c…",2025-11-07,40.495093,981,19.640321,"[{537,136}, {97,91}, {494,1038}]"
"""760c9d80-aa9f-4963-9514-34227c…",1,"""760c9d80-aa9f-4963-9514-34227c…",2025-11-03,31.648821,1214,19.640321,"[{64,328}, {106,42}, {531,110}]"
"""760c9d80-aa9f-4963-9514-34227c…",1,"""760c9d80-aa9f-4963-9514-34227c…",2025-11-06,41.515708,1069,19.640321,"[{149,219}, {12,3}, {21,187}]"


In [25]:
meals_planned = (
    menus_week
    .select(pl.col('id').alias('menu_id'), 'meals_planned')
    .explode('meals_planned')
    .unnest('meals_planned')
)

meals_planned.head()

menu_id,meal,pos
str,i64,i64
"""760c9d80-aa9f-4963-9514-34227c…",20,226
"""760c9d80-aa9f-4963-9514-34227c…",39,438
"""760c9d80-aa9f-4963-9514-34227c…",252,120
"""760c9d80-aa9f-4963-9514-34227c…",528,91
"""760c9d80-aa9f-4963-9514-34227c…",68,450


# Test the crafted menus

Check condition: (1.1), (1.2), (2), (6)

In [26]:
meal_type_vegan = dim_meal_types.filter(pl.col('meal_type_en').is_in(['vegan', 'vegetarian']))['meal_type_id']

weeklevel_idx = "34f4ad10-35c8-4cfb-a7f1-4f18c6477f3b"

(
    menus_week
    # .filter(pl.col('weeklevel_idx') == pl.lit(weeklevel_idx))

    .select('weeklevel_idx', 'date', 'meals_planned')
    .explode('meals_planned')
    .unnest('meals_planned')

    .join(
        dim_meals.select(
            'id',
            (pl.col('meal_type') == meal_type_fish).cast(pl.Int32).alias('is_fish'),
            (pl.col('meal_type').is_in(meal_type_vegan)).cast(pl.Int32).alias('is_vegan'),
            pl.col('attributes').list.contains(s_gluten).alias('is_gluten').cast(pl.Int32),
            pl.col('attributes').list.contains(s_kela).alias('is_kela').cast(pl.Int32),
        ),
        left_on='meal', right_on='id', how='left'
    )
    .group_by('weeklevel_idx', 'date')
    .agg(
        pl.len().alias('no_meals_per_day'),
        pl.col('is_fish').sum(),
        pl.col('is_vegan').sum(),
        pl.col('is_gluten').sum(),
        pl.col('is_kela').sum()
    )
    .group_by('weeklevel_idx')
    .agg(
        pl.col('no_meals_per_day').min(),
        pl.col('is_fish').sum(),
        pl.col('is_vegan').min(),
        pl.col('is_gluten').mean(),
        pl.col('is_kela').mean(),
    )

)

/var/folders/pr/8dv_cj95295bxt_hr8hzrmk40000gn/T/ipykernel_27779/1197295895.py:14: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.
  dim_meals.select(


weeklevel_idx,no_meals_per_day,is_fish,is_vegan,is_gluten,is_kela
str,u32,i32,i32,f64,f64
"""a1f00e72-64c9-488e-8b2c-8db444…",3,3,2,1.2,0.6
"""beaaf369-f6f8-41ce-8093-543875…",3,3,2,0.8,1.0
"""b24e15eb-92e2-4277-bf84-e9176f…",3,3,2,0.8,1.0
"""f3ee2231-593a-4311-9782-31c42d…",3,3,2,1.6,1.2
"""9272eadc-3c26-497f-8e82-c2dd54…",3,3,2,1.0,0.8
…,…,…,…,…,…
"""1642a60b-d314-4367-9a3a-45b395…",3,3,2,1.0,0.8
"""cc878b74-bc43-433e-b595-79b58c…",3,3,2,1.0,0.8
"""8aab0d90-4faa-4154-894f-a09745…",3,3,2,1.4,1.6


Check condition (3)

In [27]:
(
    menus_week
    # .filter(pl.col('weeklevel_idx') == pl.lit(weeklevel_idx))

    .select('weeklevel_idx', 'date', 'meals_planned')
    .explode('meals_planned')
    .unnest('meals_planned')

    .group_by('weeklevel_idx', 'meal')
    .len('no_occurrences')
    .group_by('weeklevel_idx')
    .agg(
        pl.col('no_occurrences').max().alias('no_max_occurrences')
    )

    .filter(pl.col('no_max_occurrences') > MAX_MEAL_OCCURENCES)
)

weeklevel_idx,no_max_occurrences
str,u32
